# PIDNet-S Full-HD Runtime-ROI Distance and Boundary Fine-Tuning

Fine-tune the existing Full-HD cable model with the same 16:9 crop distribution used by the flight runtime. The run keeps full-frame recovery samples, grouped validation, boundary-aware loss, independent safety checkpoints, and held-out testing.

## 1. Colab setup

Run this cell after VS Code shows a connected Colab GPU kernel. Google authentication and Drive mounting require your interaction.


In [ ]:
# Dependencies are installed once by the Colab CLI launcher before execution.
from pathlib import Path

if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Google Drive is already mounted.')


## 2. Configuration and required Drive layout

Place stand-only frames in HARD_NEGATIVE_DIR. These images must contain **no cable bundle**; the notebook assigns every pixel to background.

Optional newly annotated cable-plus-stand images can be provided as a COCO dataset under HARD_POSITIVE_ROOT with images/ and annotations.json.


In [ ]:
from pathlib import Path
import shutil

# Stage the Drive dataset on the Colab VM's local SSD. Random Full-HD reads
# directly from Drive are too slow for multi-epoch training.
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/cable_detection/mask_RCNN/augmented_dataset')
LOCAL_STAGE_ROOT = Path('/content/pidnet_fullhd_stage')
ORIGINAL_DATASET_ROOT = LOCAL_STAGE_ROOT / 'augmented_dataset'
DATASET_STAGE_MARKER = LOCAL_STAGE_ROOT / '.dataset_complete'
LOCAL_STAGE_ROOT.mkdir(parents=True, exist_ok=True)
if not DATASET_STAGE_MARKER.exists():
    print('Staging dataset from Drive to local SSD...')
    shutil.copytree(DRIVE_DATASET_ROOT, ORIGINAL_DATASET_ROOT, dirs_exist_ok=True)
    DATASET_STAGE_MARKER.write_text('complete\n')
print('Local dataset:', ORIGINAL_DATASET_ROOT)
ORIGINAL_TRAIN_IMAGES = ORIGINAL_DATASET_ROOT / 'train' / 'images'
ORIGINAL_TRAIN_COCO = ORIGINAL_DATASET_ROOT / 'train' / 'annotations.json'
ORIGINAL_TEST_IMAGES = ORIGINAL_DATASET_ROOT / 'test' / 'images'
ORIGINAL_TEST_COCO = ORIGINAL_DATASET_ROOT / 'test' / 'annotations.json'

BASE_CHECKPOINT = Path(
    '/content/drive/MyDrive/cable_detection/pidnet_s_cable_distance_boundary_finetune/checkpoints/best_safe_fullhd.pth'
)

# Optional: stand-only negatives. Training runs normally when this directory is absent.
# hard_negatives/<recording_name>/frame_000123.jpg
LOCAL_HARD_NEGATIVE_DIR = Path('/content/pidnet_hard_negatives_20260919')
DRIVE_HARD_NEGATIVE_DIR = Path(
    '/content/drive/MyDrive/cable_detection/pidnet_fullhd_hard_negatives'
)
HARD_NEGATIVE_DIR = (
    LOCAL_HARD_NEGATIVE_DIR
    if LOCAL_HARD_NEGATIVE_DIR.exists()
    else DRIVE_HARD_NEGATIVE_DIR
)

# Optional: new images that contain the real cable and stand, annotated in COCO format.
HARD_POSITIVE_ROOT = Path(
    '/content/drive/MyDrive/cable_detection/pidnet_fullhd_hard_positives'
)
HARD_POSITIVE_IMAGES = HARD_POSITIVE_ROOT / 'images'
HARD_POSITIVE_COCO = HARD_POSITIVE_ROOT / 'annotations.json'
PSEUDO_POSITIVE_ROOT = Path('/content/pidnet_pseudo_positives_20260919')
PSEUDO_POSITIVE_IMAGES = PSEUDO_POSITIVE_ROOT / 'images'
PSEUDO_POSITIVE_MASKS = PSEUDO_POSITIVE_ROOT / 'masks'

RUN_ROOT = Path(
    '/content/drive/MyDrive/cable_detection/pidnet_s_cable_runtime_roi_finetune'
)
DRIVE_MASK_CACHE = RUN_ROOT / 'semantic_mask_cache'
MASK_CACHE = LOCAL_STAGE_ROOT / 'semantic_mask_cache'
CHECKPOINT_DIR = RUN_ROOT / 'checkpoints'
REPORT_DIR = RUN_ROOT / 'reports'
for directory in (MASK_CACHE, CHECKPOINT_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
MASK_STAGE_MARKER = LOCAL_STAGE_ROOT / '.masks_complete'
if DRIVE_MASK_CACHE.exists() and not MASK_STAGE_MARKER.exists():
    print('Staging cached masks from Drive to local SSD...')
    shutil.copytree(DRIVE_MASK_CACHE, MASK_CACHE, dirs_exist_ok=True)
    MASK_STAGE_MARKER.write_text('complete\n')

TRAIN_HEIGHT = 1080
TRAIN_WIDTH = 1920
EPOCHS = 20
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 1e-4
EFFECTIVE_BATCH_SIZE = 8
VAL_FRACTION = 0.15
HARD_NEGATIVE_VAL_FRACTION = 0.20
HARD_NEGATIVE_FP_MIN_PIXELS = 1000
MAX_ALLOWED_MIOU_DROP = 0.02
EARLY_STOP_PATIENCE = 10
SEED = 42
NUM_WORKERS = 4

# Match the flight path: most samples are stand/cable ROI crops, while a
# full-frame minority preserves global recovery behavior. Crop dimensions are
# expressed relative to the source frame and retain broad stand/background
# context around the annotated cable.
RUNTIME_ROI_PROBABILITY = 0.70
RUNTIME_ROI_MIN_WIDTH_FRACTION = 0.30
RUNTIME_ROI_MAX_WIDTH_FRACTION = 0.90
RUNTIME_ROI_CONTEXT_SCALE = (2.0, 4.5)
RUNTIME_ROI_CENTER_JITTER = 0.08

CLASS_NAMES = ['background', 'Black Wire', 'Lashing Wire', 'Messenger']
NUM_CLASSES = len(CLASS_NAMES)

print('Run root:', RUN_ROOT)
print('Training resolution:', TRAIN_WIDTH, 'x', TRAIN_HEIGHT)
print('Runtime-like ROI probability:', RUNTIME_ROI_PROBABILITY)


## 3. Environment and GPU check

Batch size is intentionally conservative at Full HD. Gradient accumulation preserves an effective batch of eight.


In [ ]:
import csv
import json
import math
import os
import random
import re
import shutil
import time
from collections import defaultdict
from dataclasses import dataclass

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from pycocotools import mask as coco_mask
from pycocotools.coco import COCO
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required. In VS Code select a Colab GPU kernel.')

DEVICE = torch.device('cuda')
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
BATCH_SIZE = 2 if gpu_memory_gib >= 35 else 1
ACCUMULATION_STEPS = max(1, math.ceil(EFFECTIVE_BATCH_SIZE / BATCH_SIZE))
USE_AMP = True
PIN_MEMORY = True

print('GPU:', gpu_name)
print('GPU memory: %.1f GiB' % gpu_memory_gib)
print('Micro-batch:', BATCH_SIZE)
print('Gradient accumulation steps:', ACCUMULATION_STEPS)
print('Effective batch:', BATCH_SIZE * ACCUMULATION_STEPS)


## 4. Convert COCO annotations to semantic masks

Class IDs match the flight code. Overlap priority remains Lashing Wire, Black Wire, Messenger, then background.


In [ ]:
CLASS_PRIORITY = {0: 0, 3: 1, 1: 2, 2: 3}
NAME_TO_TRAIN_ID = {
    'black wire': 1, 'black_wire': 1, 'blackwire': 1,
    'lashing wire': 2, 'lashing_wire': 2, 'lashingwire': 2,
    'messenger': 3, 'messenger wire': 3, 'messenger_wire': 3,
}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def normalize_name(name):
    return str(name).strip().lower().replace('-', ' ').replace('__', '_')

def category_mapping(coco):
    mapping = {}
    for category in coco.loadCats(coco.getCatIds()):
        category_id = int(category['id'])
        name = normalize_name(category.get('name', category_id))
        candidates = (name, name.replace(' ', '_'), name.replace(' ', ''))
        for candidate in candidates:
            if candidate in NAME_TO_TRAIN_ID:
                mapping[category_id] = NAME_TO_TRAIN_ID[candidate]
                break
        if category_id not in mapping and category_id in (1, 2, 3):
            mapping[category_id] = category_id
    if not mapping:
        raise ValueError('No cable categories were found in the COCO file.')
    return mapping

def resolve_image_path(image_dir, file_name):
    normalized = str(file_name).replace('\\\\', '/').replace('\\', '/')
    basename = Path(normalized).name
    for candidate in (image_dir / normalized, image_dir / basename, image_dir.parent / normalized):
        if candidate.exists():
            return candidate
    matches = list(image_dir.rglob(basename))
    if matches:
        return matches[0]
    raise FileNotFoundError('Image not found for COCO entry: ' + str(file_name))

def decode_annotation(annotation, height, width):
    segmentation = annotation.get('segmentation')
    if segmentation:
        if isinstance(segmentation, list):
            rle = coco_mask.merge(coco_mask.frPyObjects(segmentation, height, width))
        elif isinstance(segmentation.get('counts'), list):
            rle = coco_mask.frPyObjects(segmentation, height, width)
        else:
            rle = segmentation
        decoded = coco_mask.decode(rle)
        if decoded.ndim == 3:
            decoded = np.any(decoded, axis=2)
        return decoded.astype(bool)
    bbox = annotation.get('bbox')
    mask = np.zeros((height, width), dtype=bool)
    if bbox:
        x, y, box_width, box_height = bbox
        x1, y1 = max(0, int(x)), max(0, int(y))
        x2 = min(width, int(math.ceil(x + box_width)))
        y2 = min(height, int(math.ceil(y + box_height)))
        mask[y1:y2, x1:x2] = True
    return mask

def source_group(file_name):
    stem = Path(str(file_name).replace('\\', '/')).stem
    match = re.match(r'^(aug_\d+)_\d+$', stem, flags=re.IGNORECASE)
    if match:
        return match.group(1).lower()
    match = re.match(r'^(.+?)[_-]frame[_-]?\d+$', stem, flags=re.IGNORECASE)
    if match:
        return match.group(1).lower()
    return stem.lower()

def convert_coco_records(split_name, image_dir, annotation_path):
    if not image_dir.exists() or not annotation_path.exists():
        raise FileNotFoundError('Missing COCO source: %s or %s' % (image_dir, annotation_path))
    coco = COCO(str(annotation_path))
    mapping = category_mapping(coco)
    cache_dir = MASK_CACHE / split_name
    cache_dir.mkdir(parents=True, exist_ok=True)
    records = []
    for image_id in tqdm(sorted(coco.getImgIds()), desc='Masks ' + split_name):
        info = coco.loadImgs([image_id])[0]
        image_path = resolve_image_path(image_dir, info['file_name'])
        mask_path = cache_dir / ('%s_%s.png' % (image_id, Path(info['file_name']).stem))
        if not mask_path.exists():
            height = int(info.get('height') or 0)
            width = int(info.get('width') or 0)
            if height <= 0 or width <= 0:
                probe = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
                if probe is None:
                    raise FileNotFoundError(image_path)
                height, width = probe.shape[:2]
            semantic = np.zeros((height, width), dtype=np.uint8)
            priorities = np.zeros((height, width), dtype=np.uint8)
            for annotation in coco.loadAnns(coco.getAnnIds(imgIds=[image_id])):
                category_id = int(annotation['category_id'])
                if category_id not in mapping:
                    continue
                train_id = mapping[category_id]
                region = decode_annotation(annotation, height, width)
                update = region & (CLASS_PRIORITY[train_id] >= priorities)
                semantic[update] = train_id
                priorities[update] = CLASS_PRIORITY[train_id]
            if not cv2.imwrite(str(mask_path), semantic):
                raise IOError('Could not save semantic mask: ' + str(mask_path))
        records.append({
            'image_path': str(image_path),
            'mask_path': str(mask_path),
            'group': source_group(info['file_name']),
            'hard_negative': False,
            'source': split_name,
        })
    return records


## 5. Build grouped training, validation, and test splits

This cell stops immediately when fewer than 50 stand-only hard negatives are available. Adjacent frames from the same recording remain in one split.


In [ ]:
for required in (
    ORIGINAL_TRAIN_IMAGES,
    ORIGINAL_TRAIN_COCO,
    ORIGINAL_TEST_IMAGES,
    ORIGINAL_TEST_COCO,
    BASE_CHECKPOINT,
):
    if not required.exists():
        raise FileNotFoundError('Required path is missing: ' + str(required))

original_records = convert_coco_records(
    'original_train', ORIGINAL_TRAIN_IMAGES, ORIGINAL_TRAIN_COCO
)
test_records = convert_coco_records(
    'original_test', ORIGINAL_TEST_IMAGES, ORIGINAL_TEST_COCO
)

hard_positive_records = []
if HARD_POSITIVE_IMAGES.exists() and HARD_POSITIVE_COCO.exists():
    hard_positive_records = convert_coco_records(
        'hard_positive', HARD_POSITIVE_IMAGES, HARD_POSITIVE_COCO
    )

# Conservative pseudo labels are generated only from fresh, temporally
# locked real-flight detections. They are training-only and are never used
# to report validation accuracy.
pseudo_positive_records = []
if PSEUDO_POSITIVE_IMAGES.exists() and PSEUDO_POSITIVE_MASKS.exists():
    for image_path in sorted(PSEUDO_POSITIVE_IMAGES.glob('*.jpg')):
        mask_path = PSEUDO_POSITIVE_MASKS / (image_path.stem + '.png')
        if not mask_path.exists():
            raise FileNotFoundError('Missing pseudo-label mask: ' + str(mask_path))
        mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        if mask is None or not np.any(mask > 0):
            raise ValueError('Empty pseudo-label mask: ' + str(mask_path))
        pseudo_positive_records.append({
            'image_path': str(image_path),
            'mask_path': str(mask_path),
            'group': source_group(image_path.name),
            'hard_negative': False,
            'source': 'pseudo_positive',
        })

hard_negative_paths = sorted(
    path for path in HARD_NEGATIVE_DIR.rglob('*')
    if HARD_NEGATIVE_DIR.exists() and path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
)
if not hard_negative_paths:
    print('No optional hard-negative images found; running annotated Full-HD fine-tuning only.')

hard_negative_records = [{
    'image_path': str(path),
    'mask_path': None,
    'group': path.parent.name.lower() if path.parent != HARD_NEGATIVE_DIR else source_group(path.name),
    'hard_negative': True,
    'source': 'hard_negative',
} for path in hard_negative_paths]

def grouped_split(records, validation_fraction, seed):
    if not records:
        return [], [], set()
    grouped = defaultdict(list)
    for record in records:
        grouped[record['group']].append(record)
    group_names = sorted(grouped)
    random.Random(seed).shuffle(group_names)
    validation_count = max(1, int(round(len(group_names) * validation_fraction)))
    validation_groups = set(group_names[:validation_count])
    train = [record for group in group_names if group not in validation_groups for record in grouped[group]]
    validation = [record for group in group_names if group in validation_groups for record in grouped[group]]
    return train, validation, validation_groups

positive_pool = original_records + hard_positive_records
positive_train, positive_validation, positive_validation_groups = grouped_split(
    positive_pool, VAL_FRACTION, SEED
)
negative_train, negative_validation, negative_validation_groups = grouped_split(
    hard_negative_records, HARD_NEGATIVE_VAL_FRACTION, SEED + 1
)

train_records = positive_train + pseudo_positive_records + negative_train
validation_records = positive_validation + negative_validation
random.Random(SEED).shuffle(train_records)

print('Positive train:', len(positive_train))
print('Positive validation:', len(positive_validation))
print('Hard-negative train:', len(negative_train))
print('Hard-negative validation:', len(negative_validation))
print('Locked real-flight pseudo-positive train:', len(pseudo_positive_records))
print('Held-out original test:', len(test_records))
print('Positive validation groups:', len(positive_validation_groups))
print('Negative validation groups:', sorted(negative_validation_groups))

train_paths = {record['image_path'] for record in train_records}
validation_paths = {record['image_path'] for record in validation_records}
assert train_paths.isdisjoint(validation_paths), 'Train/validation leakage detected'


## 6. Inspect positive and negative examples

Confirm that stand-only examples truly contain no cable. Incorrect all-background negatives will teach the network to suppress real cable pixels.


In [ ]:
def show_examples(records, title, count=4):
    chosen = records[:count]
    fig, axes = plt.subplots(1, len(chosen), figsize=(5 * len(chosen), 4))
    if len(chosen) == 1:
        axes = [axes]
    for axis, record in zip(axes, chosen):
        image = cv2.cvtColor(cv2.imread(record['image_path']), cv2.COLOR_BGR2RGB)
        axis.imshow(image)
        axis.set_title(Path(record['image_path']).name)
        axis.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

if negative_train:
    show_examples(negative_train, 'Stand-only hard negatives')
show_examples(positive_train, 'Annotated cable positives')


## 7. Runtime-matched Full-HD datasets and loaders

Training uses the exact 16:9 geometry expected by the flight ROI. Positive crops are centred near the annotated cable with randomized stand/background context, bounded centre jitter, and a 30% full-frame recovery path. Validation and held-out testing remain deterministic full-frame evaluations.

In [ ]:
def _fit_aspect_crop(center_x, center_y, crop_width, image_width, image_height, target_aspect=16.0 / 9.0):
    crop_width = float(np.clip(crop_width, 1.0, image_width))
    crop_height = crop_width / target_aspect
    if crop_height > image_height:
        crop_height = float(image_height)
        crop_width = crop_height * target_aspect
    left = float(np.clip(center_x - crop_width / 2.0, 0.0, image_width - crop_width))
    top = float(np.clip(center_y - crop_height / 2.0, 0.0, image_height - crop_height))
    return (
        int(round(left)),
        int(round(top)),
        int(round(left + crop_width)),
        int(round(top + crop_height)),
    )


def runtime_like_roi_crop(image, mask, stochastic=True):
    """Create a cable-containing crop with flight-runtime 16:9 geometry."""
    if stochastic and random.random() >= RUNTIME_ROI_PROBABILITY:
        return image, mask
    image_height, image_width = image.shape[:2]
    foreground_y, foreground_x = np.where(mask > 0)
    if foreground_x.size:
        x1, x2 = int(foreground_x.min()), int(foreground_x.max()) + 1
        y1, y2 = int(foreground_y.min()), int(foreground_y.max()) + 1
        cable_width = max(1, x2 - x1)
        cable_height = max(1, y2 - y1)
        context_scale = (
            random.uniform(*RUNTIME_ROI_CONTEXT_SCALE)
            if stochastic else float(np.mean(RUNTIME_ROI_CONTEXT_SCALE))
        )
        crop_width = max(
            cable_width * context_scale,
            cable_height * context_scale * (16.0 / 9.0),
            image_width * RUNTIME_ROI_MIN_WIDTH_FRACTION,
        )
        crop_width = min(crop_width, image_width * RUNTIME_ROI_MAX_WIDTH_FRACTION)
        center_x = 0.5 * (x1 + x2)
        center_y = 0.5 * (y1 + y2)
    else:
        if stochastic:
            crop_width = random.uniform(
                image_width * RUNTIME_ROI_MIN_WIDTH_FRACTION,
                image_width * RUNTIME_ROI_MAX_WIDTH_FRACTION,
            )
            center_x = random.uniform(0.25, 0.75) * image_width
            center_y = random.uniform(0.25, 0.75) * image_height
        else:
            crop_width = image_width * 0.60
            center_x = image_width * 0.50
            center_y = image_height * 0.50

    crop_height = min(image_height, crop_width / (16.0 / 9.0))
    if stochastic:
        center_x += random.uniform(-1.0, 1.0) * RUNTIME_ROI_CENTER_JITTER * crop_width
        center_y += random.uniform(-1.0, 1.0) * RUNTIME_ROI_CENTER_JITTER * crop_height
    crop_box = _fit_aspect_crop(
        center_x, center_y, crop_width, image_width, image_height
    )
    x1, y1, x2, y2 = crop_box

    # Jitter must never remove an annotated cable class. Fall back to a crop
    # centred exactly on the foreground when necessary.
    cropped_mask = mask[y1:y2, x1:x2]
    if foreground_x.size and not np.any(cropped_mask > 0):
        crop_box = _fit_aspect_crop(
            0.5 * (foreground_x.min() + foreground_x.max()),
            0.5 * (foreground_y.min() + foreground_y.max()),
            crop_width,
            image_width,
            image_height,
        )
        x1, y1, x2, y2 = crop_box
        cropped_mask = mask[y1:y2, x1:x2]
    return image[y1:y2, x1:x2], cropped_mask


def train_transform():
    return A.Compose([
        A.Resize(TRAIN_HEIGHT, TRAIN_WIDTH, interpolation=cv2.INTER_LINEAR),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=(-0.35, 0.10),
            rotate_limit=10,
            interpolation=cv2.INTER_LINEAR,
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.70,
        ),
        A.RandomBrightnessContrast(p=0.35),
        A.CLAHE(p=0.20),
        A.GaussianBlur(blur_limit=(3, 5), p=0.15),
        A.ImageCompression(quality_range=(55, 95), p=0.25),
        A.RGBShift(r_shift_limit=12, g_shift_limit=12, b_shift_limit=12, p=0.20),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def eval_transform():
    return A.Compose([
        A.Resize(TRAIN_HEIGHT, TRAIN_WIDTH, interpolation=cv2.INTER_LINEAR),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


class CableDataset(Dataset):
    def __init__(self, records, transform, runtime_roi=None):
        self.records = list(records)
        self.transform = transform
        if runtime_roi not in (None, 'train', 'eval'):
            raise ValueError('runtime_roi must be None, train, or eval')
        self.runtime_roi = runtime_roi

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image_bgr = cv2.imread(record['image_path'], cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise FileNotFoundError(record['image_path'])
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        if record['mask_path'] is None:
            mask = np.zeros(image.shape[:2], dtype=np.uint8)
        else:
            mask = cv2.imread(record['mask_path'], cv2.IMREAD_UNCHANGED)
            if mask is None:
                raise FileNotFoundError(record['mask_path'])
            mask = mask.astype(np.uint8)
        if self.runtime_roi:
            image, mask = runtime_like_roi_crop(
                image, mask, stochastic=self.runtime_roi == 'train'
            )
        augmented = self.transform(image=image, mask=mask)
        return augmented['image'], augmented['mask'].long(), bool(record['hard_negative'])


def make_loader(records, transform, shuffle, batch_size, runtime_roi=None):
    return DataLoader(
        CableDataset(records, transform, runtime_roi=runtime_roi),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
        drop_last=False,
    )


train_loader = make_loader(
    train_records, train_transform(), True, BATCH_SIZE, runtime_roi='train'
)
validation_loader = make_loader(
    validation_records, eval_transform(), False, BATCH_SIZE
)
runtime_validation_loader = make_loader(
    positive_validation, eval_transform(), False, BATCH_SIZE, runtime_roi='eval'
)
test_loader = make_loader(test_records, eval_transform(), False, BATCH_SIZE)

sample_images, sample_masks, sample_negative = next(iter(train_loader))
print('Image batch:', tuple(sample_images.shape))
print('Mask batch:', tuple(sample_masks.shape))
print('Mask classes:', sorted(sample_masks.unique().tolist()))

# Inspect the actual normalized runtime-ROI samples used by optimization.
preview_count = min(2, sample_images.shape[0])
fig, axes = plt.subplots(preview_count, 2, figsize=(12, 4 * preview_count), squeeze=False)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
for index in range(preview_count):
    preview = sample_images[index].permute(1, 2, 0).cpu().numpy()
    preview = np.clip(preview * std + mean, 0.0, 1.0)
    axes[index, 0].imshow(preview)
    axes[index, 0].set_title('Runtime-like training ROI')
    axes[index, 1].imshow(sample_masks[index].cpu().numpy(), vmin=0, vmax=NUM_CLASSES - 1)
    axes[index, 1].set_title('Semantic mask')
    for axis in axes[index]:
        axis.axis('off')
plt.tight_layout()
plt.show()


## 8. PIDNet-S architecture

This matches the custom architecture used by the existing checkpoint.


In [ ]:
class ConvBNAct(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=None, groups=1, activation=True):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, groups=groups, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU(inplace=True) if activation else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = ConvBNAct(in_channels, out_channels, 3, stride)
        self.conv2 = ConvBNAct(out_channels, out_channels, 3, activation=False)
        self.downsample = (
            ConvBNAct(in_channels, out_channels, 1, stride, padding=0, activation=False)
            if stride != 1 or in_channels != out_channels else nn.Identity()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv2(self.conv1(x)) + self.downsample(x))

class PagFM(nn.Module):
    def __init__(self, channels, inter_channels):
        super().__init__()
        self.f_x = ConvBNAct(channels, inter_channels, 1, padding=0, activation=False)
        self.f_y = ConvBNAct(channels, inter_channels, 1, padding=0, activation=False)

    def forward(self, x, y):
        if y.shape[-2:] != x.shape[-2:]:
            y = F.interpolate(y, size=x.shape[-2:], mode='bilinear', align_corners=False)
        sigma = torch.sigmoid(torch.sum(self.f_x(x) * self.f_y(y), dim=1, keepdim=True))
        return sigma * y + (1.0 - sigma) * x

class PAPPM(nn.Module):
    def __init__(self, in_channels, branch_channels, out_channels):
        super().__init__()
        self.scale0 = ConvBNAct(in_channels, branch_channels, 1, padding=0)
        self.scale1 = nn.Sequential(nn.AvgPool2d(5, 2, 2), ConvBNAct(in_channels, branch_channels, 1, padding=0))
        self.scale2 = nn.Sequential(nn.AvgPool2d(9, 4, 4), ConvBNAct(in_channels, branch_channels, 1, padding=0))
        self.scale3 = nn.Sequential(nn.AvgPool2d(17, 8, 8), ConvBNAct(in_channels, branch_channels, 1, padding=0))
        self.scale4 = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_channels, branch_channels, 1), nn.ReLU(inplace=True))
        self.process1 = ConvBNAct(branch_channels, branch_channels, 3)
        self.process2 = ConvBNAct(branch_channels, branch_channels, 3)
        self.process3 = ConvBNAct(branch_channels, branch_channels, 3)
        self.process4 = ConvBNAct(branch_channels, branch_channels, 3)
        self.compression = ConvBNAct(branch_channels * 5, out_channels, 1, padding=0)
        self.shortcut = ConvBNAct(in_channels, out_channels, 1, padding=0)

    def forward(self, x):
        target_size = x.shape[-2:]
        scale0 = self.scale0(x)
        scale1 = self.process1(F.interpolate(self.scale1(x), size=target_size, mode='bilinear', align_corners=False) + scale0)
        scale2 = self.process2(F.interpolate(self.scale2(x), size=target_size, mode='bilinear', align_corners=False) + scale0)
        scale3 = self.process3(F.interpolate(self.scale3(x), size=target_size, mode='bilinear', align_corners=False) + scale0)
        scale4 = self.process4(F.interpolate(self.scale4(x), size=target_size, mode='bilinear', align_corners=False) + scale0)
        return self.compression(torch.cat([scale0, scale1, scale2, scale3, scale4], dim=1)) + self.shortcut(x)

class LightBag(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.edge_gate = nn.Sequential(
            ConvBNAct(channels, channels, 3),
            nn.Conv2d(channels, channels, 1),
            nn.Sigmoid(),
        )
        self.proj = ConvBNAct(channels, channels, 3)

    def forward(self, p, i, d):
        if i.shape[-2:] != p.shape[-2:]:
            i = F.interpolate(i, size=p.shape[-2:], mode='bilinear', align_corners=False)
        if d.shape[-2:] != p.shape[-2:]:
            d = F.interpolate(d, size=p.shape[-2:], mode='bilinear', align_corners=False)
        gate = self.edge_gate(d)
        return self.proj(gate * p + (1.0 - gate) * i + d)

class SegmentHead(nn.Module):
    def __init__(self, in_channels, mid_channels, num_classes):
        super().__init__()
        self.block = nn.Sequential(
            ConvBNAct(in_channels, mid_channels, 3),
            nn.Dropout2d(0.1),
            nn.Conv2d(mid_channels, num_classes, 1),
        )

    def forward(self, x, output_size):
        return F.interpolate(self.block(x), size=output_size, mode='bilinear', align_corners=False)

class PIDNetS(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        channels = 64
        self.stem = nn.Sequential(ConvBNAct(3, 32, 3, stride=2), ConvBNAct(32, 32, 3, stride=2))
        self.layer1 = nn.Sequential(BasicBlock(32, 32), BasicBlock(32, 32))
        self.layer2 = nn.Sequential(BasicBlock(32, channels, stride=2), BasicBlock(channels, channels))
        self.p_branch = nn.Sequential(BasicBlock(channels, channels), BasicBlock(channels, channels))
        self.d_branch = nn.Sequential(BasicBlock(channels, channels), BasicBlock(channels, channels))
        self.i_layer3 = nn.Sequential(BasicBlock(channels, 128, stride=2), BasicBlock(128, 128))
        self.i_layer4 = nn.Sequential(BasicBlock(128, 256, stride=2), BasicBlock(256, 256))
        self.context = PAPPM(256, 64, channels)
        self.pag = PagFM(channels, channels // 2)
        self.bag = LightBag(channels)
        self.head = SegmentHead(channels, 128, num_classes)
        self.aux_p = SegmentHead(channels, 64, num_classes)
        self.aux_d = SegmentHead(channels, 64, num_classes)

    def forward(self, x):
        output_size = x.shape[-2:]
        x = self.layer2(self.layer1(self.stem(x)))
        p = self.p_branch(x)
        d = self.d_branch(x)
        i = self.context(self.i_layer4(self.i_layer3(x)))
        i = F.interpolate(i, size=p.shape[-2:], mode='bilinear', align_corners=False)
        p = self.pag(p, i)
        fused = self.bag(p, i, d)
        logits = self.head(fused, output_size)
        if self.training:
            return {
                'out': logits,
                'aux_p': self.aux_p(p, output_size),
                'aux_d': self.aux_d(d, output_size),
            }
        return logits


## 9. Losses and safety-aware evaluation

Class weights are computed from annotated positive masks only, preventing all-background images from collapsing the cable-class weights. Boundary-weighted cross entropy gives the thin lashing and messenger edges additional influence without globally lowering runtime confidence thresholds.


In [ ]:
def compute_positive_class_weights(records):
    counts = np.zeros(NUM_CLASSES, dtype=np.float64)
    for record in tqdm(records, desc='Class weights'):
        if record['mask_path'] is None:
            continue
        mask = cv2.imread(record['mask_path'], cv2.IMREAD_UNCHANGED)
        values, value_counts = np.unique(mask, return_counts=True)
        for value, count in zip(values, value_counts):
            if 0 <= int(value) < NUM_CLASSES:
                counts[int(value)] += int(count)
    counts = np.maximum(counts, 1.0)
    frequencies = counts / counts.sum()
    weights = 1.0 / np.log(1.02 + frequencies)
    weights = weights / weights.mean()
    return torch.tensor(np.clip(weights, 0.25, 4.0), dtype=torch.float32)

class DiceLoss(nn.Module):
    def forward(self, logits, target):
        probabilities = torch.softmax(logits, dim=1)
        one_hot = F.one_hot(target, NUM_CLASSES).permute(0, 3, 1, 2).float()
        intersection = torch.sum(probabilities * one_hot, dim=(0, 2, 3))
        cardinality = torch.sum(probabilities + one_hot, dim=(0, 2, 3))
        return 1.0 - ((2.0 * intersection + 1.0) / (cardinality + 1.0)).mean()

class CombinedLoss(nn.Module):
    def __init__(self, class_weights):
        super().__init__()
        self.register_buffer('class_weights', class_weights)
        self.cross_entropy = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = DiceLoss()

    @staticmethod
    def boundary_map(target, radius=2):
        boundary = torch.zeros_like(target, dtype=torch.bool)
        boundary[:, 1:, :] |= target[:, 1:, :] != target[:, :-1, :]
        boundary[:, :-1, :] |= target[:, :-1, :] != target[:, 1:, :]
        boundary[:, :, 1:] |= target[:, :, 1:] != target[:, :, :-1]
        boundary[:, :, :-1] |= target[:, :, :-1] != target[:, :, 1:]
        if radius > 0:
            kernel = 2 * radius + 1
            boundary = F.max_pool2d(
                boundary.float().unsqueeze(1), kernel, stride=1, padding=radius
            ).squeeze(1) > 0
        return boundary

    def boundary_cross_entropy(self, logits, target, boundary_weight=2.0):
        per_pixel = F.cross_entropy(
            logits, target, weight=self.class_weights, reduction='none'
        )
        weights = 1.0 + boundary_weight * self.boundary_map(target).float()
        return (per_pixel * weights).sum() / weights.sum().clamp_min(1.0)

    def main(self, logits, target):
        return self.boundary_cross_entropy(logits, target) + 0.5 * self.dice(logits, target)

    def forward(self, outputs, target):
        if isinstance(outputs, dict):
            return (
                self.main(outputs['out'], target)
                + 0.4 * self.cross_entropy(outputs['aux_p'], target)
                + 0.4 * self.boundary_cross_entropy(outputs['aux_d'], target)
            )
        return self.main(outputs, target)

def update_confusion(confusion, predictions, targets):
    predictions = predictions.detach().view(-1).cpu()
    targets = targets.detach().view(-1).cpu()
    labels = NUM_CLASSES * targets + predictions
    confusion += torch.bincount(labels, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)

def confusion_metrics(confusion):
    matrix = confusion.float()
    true_positive = torch.diag(matrix)
    union = matrix.sum(0) + matrix.sum(1) - true_positive
    iou = true_positive / torch.clamp(union, min=1.0)
    dice = 2.0 * true_positive / torch.clamp(matrix.sum(0) + matrix.sum(1), min=1.0)
    return {
        'mean_iou': float(iou.mean()),
        'mean_dice': float(dice.mean()),
        'pixel_accuracy': float(true_positive.sum() / torch.clamp(matrix.sum(), min=1.0)),
        'per_class_iou': {CLASS_NAMES[index]: float(iou[index]) for index in range(NUM_CLASSES)},
    }

@torch.no_grad()
def evaluate(model, loader, criterion, description):
    model.eval()
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.int64)
    loss_total = 0.0
    image_count = 0
    negative_frames = 0
    false_positive_negative_frames = 0
    negative_cable_pixels = []
    for images, masks, is_negative in tqdm(loader, desc=description):
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        with autocast('cuda', enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, masks)
        predictions = torch.argmax(logits, dim=1)
        update_confusion(confusion, predictions, masks)
        batch_count = images.shape[0]
        loss_total += float(loss) * batch_count
        image_count += batch_count
        cable_counts = (predictions > 0).flatten(1).sum(1).cpu().tolist()
        for negative, cable_count in zip(is_negative.tolist(), cable_counts):
            if negative:
                negative_frames += 1
                negative_cable_pixels.append(int(cable_count))
                false_positive_negative_frames += int(cable_count >= HARD_NEGATIVE_FP_MIN_PIXELS)
    metrics = confusion_metrics(confusion)
    metrics.update({
        'loss': loss_total / max(image_count, 1),
        'hard_negative_frames': negative_frames,
        'hard_negative_false_positive_frames': false_positive_negative_frames,
        'hard_negative_false_positive_rate': (
            false_positive_negative_frames / negative_frames if negative_frames else math.nan
        ),
        'hard_negative_median_cable_pixels': (
            float(np.median(negative_cable_pixels)) if negative_cable_pixels else math.nan
        ),
    })
    return metrics

class_weights = compute_positive_class_weights(positive_train).to(DEVICE)
criterion = CombinedLoss(class_weights)
print('Class weights:', dict(zip(CLASS_NAMES, class_weights.cpu().tolist())))


## 10. Load the existing checkpoint and establish the baseline

The baseline report proves whether the held-out stand frames reproduce the current false-positive problem before fine-tuning.


In [ ]:
def load_checkpoint(path, map_location=DEVICE):
    try:
        return torch.load(str(path), map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location=map_location)

base_checkpoint = load_checkpoint(BASE_CHECKPOINT)
model = PIDNetS(NUM_CLASSES).to(DEVICE)
model.load_state_dict(base_checkpoint['model_state'])

baseline_full_frame_metrics = evaluate(
    model, validation_loader, criterion, 'Baseline validation'
)
baseline_runtime_roi_metrics = evaluate(
    model, runtime_validation_loader, criterion, 'Baseline runtime ROI'
)
baseline_metrics = {
    'full_frame': baseline_full_frame_metrics,
    'runtime_roi': baseline_runtime_roi_metrics,
}
print(json.dumps(baseline_metrics, indent=2))
(REPORT_DIR / 'baseline_metrics.json').write_text(
    json.dumps(baseline_metrics, indent=2) + '\n'
)

if baseline_full_frame_metrics['hard_negative_frames'] == 0:
    print('No hard-negative validation set; checkpoint selection will use validation mIoU.')


## 11. Fine-tune and save independent accuracy/safety checkpoints

The safety checkpoint is selected lexicographically: lowest hard-negative false-positive rate first, then highest mIoU, while requiring mIoU to remain within MAX_ALLOWED_MIOU_DROP of the baseline.


In [ ]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LEARNING_RATE * 0.05)
scaler = GradScaler('cuda', enabled=USE_AMP)

LATEST_PATH = CHECKPOINT_DIR / 'latest_fullhd.pth'
BEST_MIOU_PATH = CHECKPOINT_DIR / 'best_miou_fullhd.pth'
BEST_SAFE_PATH = CHECKPOINT_DIR / 'best_safe_fullhd.pth'
HISTORY_PATH = REPORT_DIR / 'history.json'

def save_checkpoint(path, epoch, metrics, history):
    payload = {
        'epoch': int(epoch),
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'metrics': metrics,
        'history': history,
        'class_names': CLASS_NAMES,
        # Retain the legacy field so the existing flight loader does not crash.
        # Runtime integration must explicitly use input_height/input_width.
        'img_size': 768,
        'input_height': TRAIN_HEIGHT,
        'input_width': TRAIN_WIDTH,
        'training_resolution': [TRAIN_WIDTH, TRAIN_HEIGHT],
        'base_checkpoint': str(BASE_CHECKPOINT),
        'hard_negative_fp_min_pixels': HARD_NEGATIVE_FP_MIN_PIXELS,
        'pseudo_positive_images': len(pseudo_positive_records),
        'runtime_roi_training': {
            'probability': RUNTIME_ROI_PROBABILITY,
            'min_width_fraction': RUNTIME_ROI_MIN_WIDTH_FRACTION,
            'max_width_fraction': RUNTIME_ROI_MAX_WIDTH_FRACTION,
            'context_scale': list(RUNTIME_ROI_CONTEXT_SCALE),
            'center_jitter': RUNTIME_ROI_CENTER_JITTER,
        },
    }
    torch.save(payload, str(path))

def train_one_epoch(epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    seen = 0
    for batch_index, (images, masks, _) in enumerate(
        tqdm(train_loader, desc='Train %d/%d' % (epoch + 1, EPOCHS))
    ):
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        with autocast('cuda', enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, masks)
            scaled_loss = loss / ACCUMULATION_STEPS
        scaler.scale(scaled_loss).backward()
        should_step = (
            (batch_index + 1) % ACCUMULATION_STEPS == 0
            or batch_index + 1 == len(train_loader)
        )
        if should_step:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        running_loss += float(loss.detach()) * images.shape[0]
        seen += images.shape[0]
    return running_loss / max(seen, 1)

history = []
best_selection_score = -math.inf
best_safe_key = (math.inf, math.inf)
has_hard_negative_validation = len(negative_validation) > 0
epochs_without_safe_gain = 0
minimum_acceptable_miou = (
    baseline_full_frame_metrics['mean_iou'] - MAX_ALLOWED_MIOU_DROP
)
minimum_acceptable_runtime_roi_miou = (
    baseline_runtime_roi_metrics['mean_iou'] - 0.01
)

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(epoch)
    full_frame_metrics = evaluate(
        model, validation_loader, criterion, 'Validation %d/%d' % (epoch + 1, EPOCHS)
    )
    runtime_roi_metrics = evaluate(
        model, runtime_validation_loader, criterion,
        'Runtime ROI validation %d/%d' % (epoch + 1, EPOCHS),
    )
    selection_score = (
        0.35 * full_frame_metrics['mean_iou']
        + 0.65 * runtime_roi_metrics['mean_iou']
    )
    validation_metrics = {
        'full_frame': full_frame_metrics,
        'runtime_roi': runtime_roi_metrics,
        'selection_score': selection_score,
    }
    scheduler.step()
    row = {
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'learning_rate': optimizer.param_groups[0]['lr'],
        'full_frame_mean_iou': full_frame_metrics['mean_iou'],
        'runtime_roi_mean_iou': runtime_roi_metrics['mean_iou'],
        'selection_score': selection_score,
        'full_frame_loss': full_frame_metrics['loss'],
        'runtime_roi_loss': runtime_roi_metrics['loss'],
        'hard_negative_false_positive_rate': full_frame_metrics['hard_negative_false_positive_rate'],
        'pixel_accuracy': full_frame_metrics['pixel_accuracy'],
    }
    history.append(row)
    HISTORY_PATH.write_text(json.dumps(history, indent=2) + '\n')
    save_checkpoint(LATEST_PATH, epoch, validation_metrics, history)

    if selection_score > best_selection_score:
        best_selection_score = selection_score
        save_checkpoint(BEST_MIOU_PATH, epoch, validation_metrics, history)

    fpr = full_frame_metrics['hard_negative_false_positive_rate']
    if has_hard_negative_validation:
        safe_key = (fpr, -selection_score)
        safety_eligible = (
            math.isfinite(fpr)
            and full_frame_metrics['mean_iou'] >= minimum_acceptable_miou
            and runtime_roi_metrics['mean_iou'] >= minimum_acceptable_runtime_roi_miou
        )
    else:
        safe_key = (0.0, -selection_score)
        safety_eligible = (
            full_frame_metrics['mean_iou'] >= minimum_acceptable_miou
            and runtime_roi_metrics['mean_iou'] >= minimum_acceptable_runtime_roi_miou
        )
    if safety_eligible and safe_key < best_safe_key:
        best_safe_key = safe_key
        epochs_without_safe_gain = 0
        save_checkpoint(BEST_SAFE_PATH, epoch, validation_metrics, history)
    else:
        epochs_without_safe_gain += 1

    print(json.dumps(row, indent=2))
    if epochs_without_safe_gain >= EARLY_STOP_PATIENCE and BEST_SAFE_PATH.exists():
        print('Early stopping: no safety-checkpoint improvement.')
        break

print('Latest:', LATEST_PATH)
print('Best mIoU:', BEST_MIOU_PATH)
print('Best safety checkpoint:', BEST_SAFE_PATH if BEST_SAFE_PATH.exists() else 'NONE')


## 12. Final held-out evaluation and training curves

This cell evaluates the selected safety checkpoint on the original held-out test set and the mixed validation set. It also saves a compact report and readable training curves.


In [ ]:
if not BEST_SAFE_PATH.exists():
    raise RuntimeError(
        'No safety-eligible checkpoint was produced. Do not deploy the mIoU-only checkpoint.'
    )

selected = load_checkpoint(BEST_SAFE_PATH)
model.load_state_dict(selected['model_state'])
validation_final = evaluate(model, validation_loader, criterion, 'Selected validation')
runtime_roi_final = evaluate(
    model, runtime_validation_loader, criterion, 'Selected runtime ROI'
)
test_final = evaluate(model, test_loader, criterion, 'Original held-out test')

final_report = {
    'selected_checkpoint': str(BEST_SAFE_PATH),
    'baseline_validation': baseline_metrics,
    'selected_validation': validation_final,
    'selected_runtime_roi_validation': runtime_roi_final,
    'selected_original_test': test_final,
    'training_resolution': [TRAIN_WIDTH, TRAIN_HEIGHT],
    'hard_negative_train_images': len(negative_train),
    'hard_negative_validation_images': len(negative_validation),
    'pseudo_positive_train_images': len(pseudo_positive_records),
}
(REPORT_DIR / 'final_report.json').write_text(json.dumps(final_report, indent=2) + '\n')
print(json.dumps(final_report, indent=2))

epochs = [row['epoch'] for row in history]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(epochs, [row['full_frame_mean_iou'] for row in history], marker='o', label='Full frame')
axes[0].plot(epochs, [row['runtime_roi_mean_iou'] for row in history], marker='o', label='Runtime ROI')
axes[0].axhline(minimum_acceptable_miou, color='tab:red', linestyle='--', label='Safety floor')
axes[0].set(title='Validation mIoU', xlabel='Epoch', ylabel='mIoU')
axes[0].legend()
if has_hard_negative_validation:
    axes[1].plot(epochs, [row['hard_negative_false_positive_rate'] for row in history], marker='o')
    axes[1].set(title='Stand-only false-positive frame rate', xlabel='Epoch', ylabel='Rate')
else:
    axes[1].plot(epochs, [row['pixel_accuracy'] for row in history], marker='o')
    axes[1].set(title='Validation pixel accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[2].plot(epochs, [row['train_loss'] for row in history], label='Train')
axes[2].plot(epochs, [row['full_frame_loss'] for row in history], label='Full-frame validation')
axes[2].plot(epochs, [row['runtime_roi_loss'] for row in history], label='ROI validation')
axes[2].set(title='Loss', xlabel='Epoch', ylabel='Loss')
axes[2].legend()
for axis in axes:
    axis.grid(alpha=0.25)
plt.tight_layout()
plot_path = REPORT_DIR / 'training_summary.png'
plt.savefig(plot_path, dpi=160, bbox_inches='tight')
plt.show()
print('Saved plot:', plot_path)


## 13. Next steps

1. Inspect final_report.json and training_summary.png.
2. Replay every raw flight recording with best_safe_fullhd.pth.
3. Require zero stable stand-only locks, not merely improved mIoU.
4. Update the live loader to read input_height and input_width before enabling native Full-HD inference.
5. Keep the vertical acquisition ROI and bottom-edge rejection as independent safety gates.
6. Only copy the selected checkpoint into the live checkpoint location after offline video acceptance tests pass.
